# DNS Tunneling Detection — DistilBERT (Colab)

End-to-end notebook: clone repo, install dependencies, load data, fine-tune DistilBERT, evaluate, compare with classical ML.

**Repo**: https://github.com/df-DNS-Tunneling-Detection/DNS_Tunneling_Detection  
**Open in Colab**: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/df-DNS-Tunneling-Detection/DNS_Tunneling_Detection/blob/main/llm/04_distilbert_e2e.ipynb)

## How to run this notebook

> **Estimated time: ~30-60 minutes** on a free T4 GPU runtime.  
> **Required**: T4 GPU (Runtime → Change runtime type → T4 GPU)

1. Open in Colab (badge above).
2. *(Optional)* `File → Save a copy in Drive` to persist your edits.
3. Run **section 1** to clone the repo and install dependencies.
4. Run **section 2** to load the dataset (bundled sample or your own).
5. Run **section 3** for quick EDA.
6. Run **section 4** to split and tokenize the data.
7. Run **section 5** to fine-tune DistilBERT (~30-60 min on T4).
8. Run **section 6** to evaluate DistilBERT on the test set.
9. Run **section 7** to train and evaluate classical ML models (RF + XGBoost).
10. Run **section 8** for side-by-side model comparison and all plots.
11. Run **section 9** for inference demo on sample queries.

### Troubleshooting

| Symptom | Fix |
|---|---|
| `RuntimeError: CUDA out of memory` | Reduce `BATCH_SIZE` to 16 or 32. |
| `ModuleNotFoundError: transformers` | Re-run section 1. |
| `FileNotFoundError: .../sample.csv` | Re-run section 1 to clone the repo. |
| Training is very slow | Make sure T4 GPU is enabled (Runtime → Change runtime type). |
| `torch not compiled with CUDA` | You are on CPU. Switch to T4 GPU runtime. |

## 1. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/df-DNS-Tunneling-Detection/DNS_Tunneling_Detection.git
%cd DNS_Tunneling_Detection/llm

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn xgboost seaborn joblib matplotlib pandas numpy tqdm

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## 2. Configuration and load data

In [ ]:
from pathlib import Path

DATA_PATH = Path("../data/raw/sample.csv")
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 64
EPOCHS = 3
LEARNING_RATE = 2e-5
OUTPUT_DIR = Path("models/distilbert")
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL2ID = {"benign": 0, "tunneling": 1}
ID2LABEL = {0: "benign", 1: "tunneling"}

### Load dataset

Uses the bundled synthetic sample by default. To use your own data, upload a CSV to `data/raw/` and update `DATA_PATH` above.

In [ ]:
import pandas as pd

if not DATA_PATH.exists():
    !python -m src.generate_sample --out {DATA_PATH}

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["query", "label"])
df["query"] = df["query"].astype(str)
df["label"] = df["label"].astype(int)

print(f"Total samples: {len(df):,}")
print(f"Benign:    {(df['label'] == 0).sum():,}")
print(f"Tunneling: {(df['label'] == 1).sum():,}")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

counts = df["label"].value_counts().rename({0: "benign", 1: "tunneling"})
counts.plot(kind="bar", ax=axes[0], color=["#4c72b0", "#dd8452"])
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=0)

df["query_length"] = df["query"].str.len()
for label, name, color in [(0, "benign", "#4c72b0"), (1, "tunneling", "#dd8452")]:
    subset = df[df["label"] == label]
    axes[1].hist(subset["query_length"], bins=50, alpha=0.7, label=name, color=color)
axes[1].set_title("Query Length Distribution")
axes[1].set_xlabel("Length (chars)")
axes[1].legend()

from math import log2
from collections import Counter

def shannon_entropy(s):
    if not s:
        return 0.0
    counts = Counter(s)
    n = len(s)
    return -sum((c / n) * log2(c / n) for c in counts.values())

df["entropy"] = df["query"].apply(shannon_entropy)
for label, name, color in [(0, "benign", "#4c72b0"), (1, "tunneling", "#dd8452")]:
    subset = df[df["label"] == label]
    axes[2].hist(subset["entropy"], bins=50, alpha=0.7, label=name, color=color)
axes[2].set_title("Shannon Entropy Distribution")
axes[2].set_xlabel("Entropy (bits)")
axes[2].legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_overview.png", dpi=150)
plt.show()

In [ ]:
print("=== Sample Benign Queries ===")
for q in df[df["label"] == 0]["query"].sample(5, random_state=1).tolist():
    print(f"  {q}")

print("\n=== Sample Tunneling Queries ===")
for q in df[df["label"] == 1]["query"].sample(5, random_state=1).tolist():
    print(f"  {q}")

## 4. Split and tokenize

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

print(f"Train: {len(train_df):,}")
print(f"Val:   {len(val_df):,}")
print(f"Test:  {len(test_df):,}")

test_df.to_csv("test_split.csv", index=False)

In [ ]:
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts):
    return tokenizer(
        texts.tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

train_enc = tokenize(train_df["query"])
val_enc = tokenize(val_df["query"])
test_enc = tokenize(test_df["query"])

print(f"Tokenized {len(train_df):,} train / {len(val_df):,} val / {len(test_df):,} test")
print(f"Input IDs shape: {train_enc['input_ids'].shape}")

In [ ]:
class DNSDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_ds = DNSDataset(train_enc, torch.tensor(train_df["label"].values))
val_ds = DNSDataset(val_enc, torch.tensor(val_df["label"].values))
test_ds = DNSDataset(test_enc, torch.tensor(test_df["label"].values))

## 5. Fine-tune DistilBERT

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    fp16=(device == "cuda"),
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Model saved to {OUTPUT_DIR}")

## 6. Evaluate DistilBERT on test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, precision_recall_curve

predictions = trainer.predict(test_ds)
db_probs = torch.softmax(torch.tensor(predictions.predictions), dim=-1).numpy()[:, 1]
db_preds = np.argmax(predictions.predictions, axis=-1)
db_labels = test_df["label"].values

print(classification_report(db_labels, db_preds, target_names=["benign", "tunneling"]))

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
cm = confusion_matrix(db_labels, db_preds)
ConfusionMatrixDisplay(cm, display_labels=["benign", "tunneling"]).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — DistilBERT")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "confusion_distilbert.png", dpi=150)
plt.show()

## 7. Classical ML models (Random Forest & XGBoost)

In [ ]:
import sys
sys.path.insert(0, str(Path("..").resolve()))

from src.features import extract_features

train_classical, test_classical = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

X_train_cl = extract_features(train_classical["query"])
y_train_cl = train_classical["label"].values
X_test_cl = extract_features(test_classical["query"])
y_test_cl = test_classical["label"].values

print(f"Classical features: {list(X_train_cl.columns)}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42, class_weight="balanced")
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, eval_metric="logloss", tree_method="hist", n_jobs=-1, random_state=42)

rf.fit(X_train_cl, y_train_cl)
xgb.fit(X_train_cl, y_train_cl)

rf_preds = rf.predict(X_test_cl)
rf_probs = rf.predict_proba(X_test_cl)[:, 1]

xgb_preds = xgb.predict(X_test_cl)
xgb_probs = xgb.predict_proba(X_test_cl)[:, 1]

print("Random Forest:")
print(classification_report(y_test_cl, rf_preds, target_names=["benign", "tunneling"]))
print("XGBoost:")
print(classification_report(y_test_cl, xgb_preds, target_names=["benign", "tunneling"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

cm_rf = confusion_matrix(y_test_cl, rf_preds)
ConfusionMatrixDisplay(cm_rf, display_labels=["benign", "tunneling"]).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix — Random Forest")

cm_xgb = confusion_matrix(y_test_cl, xgb_preds)
ConfusionMatrixDisplay(cm_xgb, display_labels=["benign", "tunneling"]).plot(ax=axes[1], colorbar=False, cmap="Blues")
axes[1].set_title("Confusion Matrix — XGBoost")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "confusion_classical.png", dpi=150)
plt.show()

## 8. Model comparison

In [ ]:
def get_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }

db_metrics = get_metrics(db_labels, db_preds, db_probs)
rf_metrics = get_metrics(y_test_cl, rf_preds, rf_probs)
xgb_metrics = get_metrics(y_test_cl, xgb_preds, xgb_probs)

comparison = pd.DataFrame({
    "DistilBERT": db_metrics,
    "Random Forest": rf_metrics,
    "XGBoost": xgb_metrics,
}).T.round(4)

comparison.to_csv("metrics_comparison.csv")
print(comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

metrics_to_plot = ["accuracy", "precision", "recall", "f1", "roc_auc"]
x = np.arange(len(metrics_to_plot))
width = 0.25
colors = ["#55a868", "#4c72b0", "#dd8452"]

for i, (name, metrics) in enumerate({"DistilBERT": db_metrics, "Random Forest": rf_metrics, "XGBoost": xgb_metrics}.items()):
    vals = [metrics[m] for m in metrics_to_plot]
    ax.bar(x + i * width, vals, width, label=name, color=colors[i])

ax.set_xticks(x + width)
ax.set_xticklabels([m.replace("_", "-").upper() for m in metrics_to_plot])
ax.set_ylim(0.9, 1.01)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — All Metrics")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "metric_comparison_bars.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for name, y_true, y_prob, color in [
    ("DistilBERT", db_labels, db_probs, "#55a868"),
    ("Random Forest", y_test_cl, rf_probs, "#4c72b0"),
    ("XGBoost", y_test_cl, xgb_probs, "#dd8452"),
]:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc:.4f})", color=color)

ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC — Model Comparison")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "roc_comparison_all.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for name, y_true, y_prob, color in [
    ("DistilBERT", db_labels, db_probs, "#55a868"),
    ("Random Forest", y_test_cl, rf_probs, "#4c72b0"),
    ("XGBoost", y_test_cl, xgb_probs, "#dd8452"),
]:
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = np.trapz(prec, rec)
    ax.plot(rec, prec, label=f"{name} (PR-AUC = {pr_auc:.4f})", color=color)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall — Model Comparison")
ax.legend(loc="lower left")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "pr_comparison_all.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, model_obj, name in [(axes[0], rf, "Random Forest"), (axes[1], xgb, "XGBoost")]:
    importances = model_obj.feature_importances_
    order = np.argsort(importances)[::-1]
    sns.barplot(x=importances[order], y=np.array(X_train_cl.columns)[order], ax=ax, palette="viridis")
    ax.set_title(f"Feature Importance — {name}")
    ax.set_xlabel("Importance")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "feature_importance_classical.png", dpi=150)
plt.show()

## 9. Inference demo

In [ ]:
sample_queries = [
    "mail.google.com",
    "api.github.com",
    "aXk1bG9yZW0gaXBzdW0gZG9sb3Igc2l0.tunnel.evil.com",
    "c2VjcmV0LWRhdGEtZXhmaWx0cmF0aW9u.payload.attacker.io",
    "cdn.amazon.com",
    "7su1gouejhsw7ygwi8zip0nvzefcmvcpxle5ak58w6.payload.evil.com",
    "www.stackoverflow.com",
    "r2cx6uvjplhlexn5rorozbmadn8s8pt0wxrxkacjrheewciahrr6.exfil.evil.org",
]

enc = tokenizer(sample_queries, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()

print(f"{'QUERY':<75} {'PRED':>10} {'CONF':>8} {'P_TUNNEL':>10}")
print("-" * 107)
for q, prob in zip(sample_queries, probs):
    pred = "tunneling" if prob[1] >= 0.5 else "benign"
    conf = prob.max()
    print(f"{q:<75} {pred:>10} {conf:>8.4f} {prob[1]:>10.4f}")

## 10. Download results

In [ ]:
from google.colab import files

!zip -r distilbert_model.zip models/distilbert/
!zip -r figures.zip figures/

print("Files ready for download:")
print("  - distilbert_model.zip (trained model)")
print("  - figures.zip (all evaluation plots)")
print("  - metrics_comparison.csv (comparison table)")
print("  - test_split.csv (test data)")
print("\nDownload from the file pane on the left, or run:")
print("  files.download('distilbert_model.zip')")
print("  files.download('figures.zip')")
print("  files.download('metrics_comparison.csv')")

## Summary

| Aspect | Classical ML (RF/XGBoost) | DistilBERT |
|---|---|---|
| Input | 11 hand-crafted features | Raw query text |
| Feature engineering | Required | None |
| Training time | Seconds | ~30-60 min (T4) |
| Inference speed | 1000s/sec | ~100/sec |
| Interpretability | High (feature importance) | Low (black box) |
| Deployment | CPU-friendly | Needs GPU or optimization |